
# HerWILL Safe Social Media Datathon 2026 — Harmful Content Classification
## "Best result" version: multi-model ensemble

This is the **push-to-the-ceiling** version of the pipeline. Compared to the fast
baseline notebook, it adds:

1. **`xlm-roberta-large`** instead of base (usually +2-4 macro-F1 points, larger/slower).
2. **A second, architecturally-different multilingual encoder — `microsoft/mdeberta-v3-base`**
   — ensembled with XLM-R by averaging softmax probabilities. mDeBERTa-v3 is a strong,
   genuinely multilingual model (not Bangla-only), so it stays strong on the ~58% of
   rows that are English-only, unlike a Bangla-specialist model would.
3. **A slightly higher weight on class 2 (Neutral)**, the rarest class (~12% of data),
   on top of the standard "balanced" class weights.
4. Same data-cleaning and train/test-leakage handling as the fast baseline (already
   validated against your files): dedup, conflicting-label majority vote, outlier
   drop, exact-match override.

**Trade-off — this is slower.** Rough Kaggle GPU-time budget with the default
"best" config below (`xlm-roberta-large`: 3 folds x 3 epochs, `mdeberta-v3-base`:
5 folds x 4 epochs) is roughly **9-12 hours** on a single T4/P100. Kaggle sessions
cap at 12h and give ~30 GPU-hours/week, so this fits in one sitting but leaves
little margin. If you're short on quota, flip `BUDGET_MODE` to `"fast"` below to
fall back to the single xlm-roberta-base config from the baseline notebook (~1-2h).

**Expected macro F1**: baseline notebook ≈ 0.72-0.78; this version ≈ 0.78-0.83,
depending on how label noise in the data plays out on the private test set (see the
"Realistic ceiling" discussion — ~111 rows in training have genuine annotator
disagreement on the same text, which caps how far *any* model can go).

> **Before running on Kaggle:** add your dataset, edit `DATA_DIR` below, and turn on
> GPU + Internet in Notebook Settings.


## 1. Setup

In [ ]:

import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("transformers")
ensure("datasets")
ensure("scikit-learn", "sklearn")
ensure("sentencepiece")
ensure("protobuf")


In [ ]:

import os, re, gc, random, unicodedata, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. This notebook needs a GPU to finish in reasonable time.")


## 2. Config — EDIT `DATA_DIR`, choose `BUDGET_MODE`

In [ ]:

# --------------------------------------------------------------------------
# EDIT THIS to match the path of your dataset on Kaggle, e.g.
#   "/kaggle/input/herwill-safe-social-media-datathon-2026"
# --------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/herwill-safe-social-media-datathon-2026"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_LABELS = 3
MAX_CHARS_TRAIN = 2000   # drop training rows longer than this (scraped-thread outliers)
CLASS2_WEIGHT_MULTIPLIER = 1.3  # extra boost for the rare Neutral class, on top of "balanced"

# "best"  -> xlm-roberta-large + mdeberta-v3-base ensemble (slower, higher ceiling)
# "fast"  -> single xlm-roberta-base run (same as the baseline notebook, ~1-2h)
BUDGET_MODE = "best"

if BUDGET_MODE == "best":
    MODEL_CONFIGS = [
        dict(name="xlm-roberta-large", n_folds=3, epochs=3, lr=1e-5,
             batch_size=16, eval_batch_size=32, max_len=128, warmup_ratio=0.06, weight_decay=0.01),
        dict(name="microsoft/mdeberta-v3-base", n_folds=5, epochs=4, lr=2e-5,
             batch_size=32, eval_batch_size=64, max_len=128, warmup_ratio=0.06, weight_decay=0.01),
    ]
elif BUDGET_MODE == "fast":
    MODEL_CONFIGS = [
        dict(name="xlm-roberta-base", n_folds=5, epochs=3, lr=2e-5,
             batch_size=32, eval_batch_size=64, max_len=128, warmup_ratio=0.06, weight_decay=0.01),
    ]
else:
    raise ValueError("BUDGET_MODE must be 'best' or 'fast'")

for cfg in MODEL_CONFIGS:
    print(cfg)

assert os.path.exists(TRAIN_PATH), f"train.csv not found at {TRAIN_PATH} -- fix DATA_DIR above."
assert os.path.exists(TEST_PATH), f"test.csv not found at {TEST_PATH} -- fix DATA_DIR above."
print("Found train.csv and test.csv OK.")


## 3. Load Data

In [ ]:

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()



## 4. Data Cleaning

Same issues found and fixed as in the baseline notebook (verified against your
actual `train.csv` / `test.csv`):

| Problem | Fix |
|---|---|
| 576 exact duplicate `(text, y)` rows | dropped |
| 111 rows from texts with conflicting labels (annotator disagreement) | majority vote; true ties dropped |
| A few extreme outlier rows (scraped multi-comment threads mislabeled as class 0) | dropped from training only, `text.length > 2000` chars |
| Whitespace / unicode noise | NFC normalize + collapse whitespace |
| 272 identical posts shared between train & test | after modeling, reuse the known train label for those specific test rows |

We deliberately do **not** machine-translate the Bangla/English/Banglish text — see
the baseline notebook for the reasoning (translation flattens exactly the
sarcasm/code-switch signal this task is testing). Both models here are multilingual
and handle code-mixed input directly.


In [ ]:

def normalize_text(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u200c", "").replace("\u200d", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_train(df: pd.DataFrame, max_chars: int = MAX_CHARS_TRAIN) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].apply(normalize_text)

    before = len(df)
    df = df.drop_duplicates(subset=["text", "y"], keep="first")
    print(f"Dropped {before - len(df)} exact duplicate (text,y) rows.")

    def majority_label(s):
        vc = s.value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]:
            return vc.idxmax()
        return np.nan

    label_map = df.groupby("text")["y"].apply(majority_label)
    before = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first").copy()
    df["y"] = df["text"].map(label_map)
    df = df[df["y"].notna()].reset_index(drop=True)
    df["y"] = df["y"].astype(int)
    print(f"Resolved/dropped {before - len(df)} rows from duplicate texts (majority vote or tie-drop).")

    before = len(df)
    df = df[df["text"].str.len() <= max_chars].reset_index(drop=True)
    print(f"Dropped {before - len(df)} extreme-outlier rows (> {max_chars} chars).")

    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df

def clean_test(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].apply(normalize_text)
    df["text"] = df["text"].replace("", "[EMPTY]")
    return df

train_clean = clean_train(train_df)
test_clean = clean_test(test_df)

print("\nFinal train shape:", train_clean.shape)
print(train_clean["y"].value_counts())


## 5. Dataset class (tokenizer supplied per model)

In [ ]:

class ToxicityDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


## 6. Metric + Class-Weighted Trainer

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

class WeightedTrainer(Trainer):
    '''Trainer with a class-weighted CrossEntropy loss to counter the
    imbalance between Explicit (0), Subtle (1) and Neutral (2).'''
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def make_training_args(out_dir, cfg):
    kwargs = dict(
        output_dir=out_dir,
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["eval_batch_size"],
        num_train_epochs=cfg["epochs"],
        learning_rate=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        warmup_ratio=cfg["warmup_ratio"],
        logging_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    try:
        return TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **kwargs)



## 7. Train every model x fold, accumulate OOF + test probabilities

Both models share the exact same `StratifiedKFold` splits (same `SEED`), so their
out-of-fold (OOF) probabilities can be summed index-for-index and averaged into one
honest ensemble estimate — this is what tells us, before touching the leaderboard,
roughly where the ensemble actually lands.


In [ ]:

texts = train_clean["text"].values
labels = train_clean["y"].astype(int).values
n_train = len(labels)
n_test = len(test_clean)

oof_probs_sum = np.zeros((n_train, NUM_LABELS), dtype=np.float64)
test_probs_sum = np.zeros((n_test, NUM_LABELS), dtype=np.float64)
n_models_in_ensemble = 0
per_model_oof_f1 = {}

for cfg in MODEL_CONFIGS:
    model_name = cfg["name"]
    n_folds = cfg["n_folds"]
    print(f"\n{'#'*70}\nMODEL: {model_name}  ({n_folds} folds, {cfg['epochs']} epochs, lr={cfg['lr']})\n{'#'*70}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    test_dataset = ToxicityDataset(test_clean["text"].values, tokenizer, labels=None, max_len=cfg["max_len"])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    model_oof_preds = np.zeros(n_train, dtype=int)
    model_test_probs_sum = np.zeros((n_test, NUM_LABELS), dtype=np.float64)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
        print(f"\n--- {model_name} | fold {fold + 1}/{n_folds} ---")

        X_train, y_train = texts[train_idx], labels[train_idx]
        X_val, y_val = texts[val_idx], labels[val_idx]

        train_ds = ToxicityDataset(X_train, tokenizer, y_train, max_len=cfg["max_len"])
        val_ds = ToxicityDataset(X_val, tokenizer, y_val, max_len=cfg["max_len"])

        class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_LABELS), y=y_train)
        class_weights[2] *= CLASS2_WEIGHT_MULTIPLIER
        class_weights = torch.tensor(class_weights, dtype=torch.float)
        print("Class weights:", class_weights.tolist())

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS)

        safe_name = model_name.replace("/", "_")
        fold_dir = os.path.join(OUTPUT_DIR, f"{safe_name}_fold_{fold}")
        args = make_training_args(fold_dir, cfg)

        trainer = WeightedTrainer(
            class_weights=class_weights,
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=data_collator,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
        )
        trainer.train()

        val_pred = trainer.predict(val_ds)
        val_probs = torch.softmax(torch.tensor(val_pred.predictions), dim=1).numpy()
        oof_probs_sum[val_idx] += val_probs
        val_argmax = np.argmax(val_probs, axis=1)
        model_oof_preds[val_idx] = val_argmax
        fold_f1 = f1_score(y_val, val_argmax, average="macro")
        fold_scores.append(fold_f1)
        print(f"{model_name} fold {fold + 1} macro F1: {fold_f1:.4f}")

        test_pred = trainer.predict(test_dataset)
        test_probs = torch.softmax(torch.tensor(test_pred.predictions), dim=1).numpy()
        model_test_probs_sum += test_probs

        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    model_oof_f1 = f1_score(labels, model_oof_preds, average="macro")
    per_model_oof_f1[model_name] = model_oof_f1
    print(f"\n>>> {model_name} overall OOF macro F1: {model_oof_f1:.4f}")
    print(f"    per-fold: {[round(s,4) for s in fold_scores]}")

    test_probs_sum += model_test_probs_sum / n_folds
    n_models_in_ensemble += 1

print("\n" + "=" * 70)
print("Per-model OOF macro F1:", {k: round(v, 4) for k, v in per_model_oof_f1.items()})


## 8. Ensemble OOF score (honest estimate before submitting)

In [ ]:

oof_probs_avg = oof_probs_sum / n_models_in_ensemble
oof_preds_ensemble = np.argmax(oof_probs_avg, axis=1)
ensemble_oof_f1 = f1_score(labels, oof_preds_ensemble, average="macro")

print(f"ENSEMBLE OOF macro F1: {ensemble_oof_f1:.4f}\n")
print(classification_report(labels, oof_preds_ensemble, target_names=["Explicit(0)", "Subtle(1)", "Neutral(2)"]))


## 9. Final Test Predictions + Leakage Override

In [ ]:

test_probs_avg = test_probs_sum / n_models_in_ensemble
final_preds = np.argmax(test_probs_avg, axis=1)

submission = pd.DataFrame({
    "id": test_clean["id"].values,
    "y_pred": final_preds,
})

text_to_label = dict(zip(train_clean["text"], train_clean["y"].astype(int)))
test_text_lookup = test_clean.set_index("id")["text"].to_dict()

overridden = 0
for i, row_id in enumerate(submission["id"]):
    t = test_text_lookup[row_id]
    if t in text_to_label:
        new_label = text_to_label[t]
        if submission.loc[i, "y_pred"] != new_label:
            overridden += 1
        submission.loc[i, "y_pred"] = new_label

print(f"Overrode {overridden} predictions using exact train/test text matches.")


## 10. Save Submission

In [ ]:

assert submission["id"].isin(test_df["id"]).all(), "Some ids don't match test.csv!"
assert set(submission["id"]) == set(test_df["id"]), "Missing/extra ids vs test.csv!"
assert submission["y_pred"].isin([0, 1, 2]).all(), "Found a label outside {0,1,2}!"
assert submission.shape[0] == test_df.shape[0], "Row count mismatch with test.csv!"

submission = submission.set_index("id").loc[test_df["id"]].reset_index()

submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.head())
print("\nPredicted label distribution on test set:")
print(submission["y_pred"].value_counts())



## Where the ceiling actually is

The `ENSEMBLE OOF macro F1` printed in Section 8 is your best honest estimate of
where this data + this approach lands — trust it more than intuition about "what a
good score should be."

**If OOF is meaningfully below ~0.78-0.80**, the bottleneck is very likely label
quality, not the model:
- Re-check the ~111 conflicting-label rows this notebook resolved automatically —
  manually relabeling the clearest mistakes among them can move class-2 F1 more than
  another hour of training.
- Look at the per-class report in Section 8. If class 1 (Subtle) precision is the
  weak point, that's expected — sarcasm/coded language is the hardest category for
  *any* system, human annotators included, which is exactly why the competition
  emphasizes it.

**If you still have GPU budget left after this run**, the next-highest-leverage
additions, roughly in order:
1. A 3rd ensemble member for extra diversity (e.g. `bert-base-multilingual-cased` or
   `xlm-roberta-base` at a different `MAX_LEN`), same recipe as above.
2. A short learning-rate sweep per model (`{1e-5, 2e-5, 3e-5}`) picking by OOF F1
   instead of the fixed defaults used here.
3. Increasing `MAX_LEN` to 192-256 if truncation is cutting off real (non-outlier)
   posts — check `train_clean["text"].str.len().quantile(0.99)` to see if 128 is
   actually enough.
